# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's daily performance record, at grain report_date × client_hash_id × content_hash_id. Verified with a full duplicate-group count, not a limited preview: 0 duplicate groups, 0 duplicate rows, max group size = 1 across all 9,841,378 rows — the grain claim holds exactly, not just plausibly. PASS.

Time window: month=2026-03, spanning 2026-03-01 to 2026-03-31, with 31 distinct calendar days confirmed against the 31 expected days in March — no gaps in the middle of the month. PASS (31/31 days present). This is deliberately a mid-panel month, not _sample (June 2026, the sealed final month / natural outcome window for any label).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Grain probe: full duplicate count, not a LIMIT-ed preview
grain_check = con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE c > 1) AS duplicate_groups,
        COALESCE(SUM(c) FILTER (WHERE c > 1), 0) AS duplicate_rows,
        COALESCE(MAX(c), 0) AS max_group_size
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM read_parquet('{TABLE}')
        GROUP BY report_date, client_hash_id, content_hash_id
    )
""").df()
print("Grain check (full count, no LIMIT):")
print(grain_check)
print("PASS" if grain_check["duplicate_groups"].iloc[0] == 0 else "FAIL")

# Window check: min/max PLUS distinct day count vs expected 31
window_check = con.sql(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS distinct_days,
        COUNT(*) AS row_count
    FROM read_parquet('{TABLE}')
""").df()
print("\nDate span + row count for this partition:")
print(window_check)
print("PASS (31/31 days present)" if window_check["distinct_days"].iloc[0] == 31 else "FAIL (missing days)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (full count, no LIMIT):
   duplicate_groups  duplicate_rows  max_group_size
0                 0             0.0               1
PASS

Date span + row count for this partition:
    min_date   max_date  distinct_days  row_count
0 2026-03-01 2026-03-31             31    9841378
PASS (31/31 days present)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Lane 3's clustering dataframe is built by joining dim_content onto an aggregated monthly slice of fact_content_daily_performance. Aggregation grain after joining: one row per content_hash_id × client_hash_id × month — the daily report_date grain is collapsed via aggregation (sum/mean over March) before clustering; dim_content's static fields attach once per content_hash_id.

Label: None is used for model fitting — Lane 3 is unsupervised clustering with no supervised target. Any cluster ID, archetype name, or recommended action produced later is a downstream interpretation output, not a label and never fed back in as a feature.

Caveat on the join, corrected from an earlier overstatement: a prior check showed 9,841,378 fact rows all matching a content_hash_id in dim_content — this only proves the fact→dim direction has no unmatched keys. It does not prove dim_content has no duplicate content_hash_id values (which would silently fan out fact rows while still showing "matched"), and it says nothing about dim_content rows with zero fact-side matches (expected — not every page had March activity). Both need a dedicated check in Section 3, not inferred from the match count alone.

# Field Inventory

## 1. Join Keys / Grain

### `content_hash_id`
- **Source:** both
- **Bucket:** join key / grain
- **Role / use:** joins the two tables; part of clustering grain
- **QA status:** —
- **Decision / reason:** not a feature

### `client_hash_id`
- **Source:** both
- **Bucket:** grain / split
- **Role / use:** part of grain; also used for client-holdout-style splits if needed later
- **QA status:** —
- **Decision / reason:** not a feature

### `report_date`
- **Source:** fact
- **Bucket:** grain (pre-aggregation)
- **Role / use:** collapsed away once aggregated to the month
- **QA status:** —
- **Decision / reason:** not a feature after aggregation

---

## 2. Identifier-only Fields

### `keyword_hash_id`, `url_hash_id`
- **Source:** dim_content
- **Bucket:** identifier-only
- **Role / use:** no clustering use identified
- **QA status:** —
- **Decision / reason:** excluded — identifier-only

---

## 3. Filters / QA Fields

### `month`
- **Source:** fact
- **Bucket:** filter / QA
- **Role / use:** confirms correct partition selected
- **QA status:** —
- **Decision / reason:** not a feature

### `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`
- **Source:** fact
- **Bucket:** filter / QA
- **Role / use:** used with IS TRUE to select valid rows before clustering
- **QA status:** —
- **Decision / reason:** excluded — availability flag, not behavior

### `is_published`, `is_deleted`
- **Source:** dim_content
- **Bucket:** filter / QA
- **Role / use:** likely used to exclude deleted/unpublished pages before clustering
- **QA status:** needs null/value check
- **Decision / reason:** excluded — status flag, not behavior

---

## 4. Candidate Inputs

### `content_type`
- **Source:** dim_content
- **Bucket:** candidate input
- **Role / use:** 3 categories, verified 0% null, sums exactly to 519,606
- **QA status:** verified clean
- **Decision / reason:** candidate for final 5-feature frame

### `word_count`
- **Source:** dim_content
- **Bucket:** candidate input
- **Role / use:** content shape signal
- **QA status:** verified 34.2% null
- **Decision / reason:** candidate, requires has_word_count flag — cannot fillna(0) blindly

### `search_volume`, `competition`
- **Source:** dim_content
- **Bucket:** candidate input
- **Role / use:** search-context signals
- **QA status:** verified 27.4% null, identical rate — correlated missingness
- **Decision / reason:** candidate, requires shared has_search_data flag

### `main_intent`, `char_count`, `backlinks`, `category_count`, `competition_level`, `cpc`
- **Source:** dim_content
- **Bucket:** candidate input
- **Role / use:** possible behavioral/structural signals
- **QA status:** not yet null-checked
- **Decision / reason:** still open — QA needed in Section 3 before inclusion decision

### `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic/direct/referral/social/paid/ai`, `scroll_events`
- **Source:** fact
- **Bucket:** candidate inputs considered
- **Role / use:** behavioral signals aggregated over March
- **QA status:** not yet null-checked
- **Decision / reason:** narrowed down to the actual 5 chosen features in Section 3, not all used

---

## 5. Derived Intermediate Fields

### `content_created_date`, `content_updated_date`
- **Source:** dim_content
- **Bucket:** derived intermediate
- **Role / use:** used to compute content_age_days, not clustered on raw
- **QA status:** —
- **Decision / reason:** intermediate only, not a direct feature

---

## 6. Excluded Fields

### `gsc_sum_position`
- **Source:** fact
- **Bucket:** excluded
- **Role / use:** duplicate/proxy of gsc_avg_position; not comparable across content with different observed-day counts
- **QA status:** —
- **Decision / reason:** excluded — duplicate/proxy of another field

### `keyword_char_count`, `keyword_token_count`, `url_char_count`, `keyword_created_date`
- **Source:** dim_content
- **Bucket:** excluded
- **Role / use:** describe the keyword/URL, not page behavior
- **QA status:** —
- **Decision / reason:** excluded — out of scope for a behavioral archetype

### `provider_used`, `model_used`
- **Source:** dim_content
- **Bucket:** excluded
- **Role / use:** describes generation method, not performance
- **QA status:** —
- **Decision / reason:** excluded — out of scope

### `optimization_eligible_date`, `last_optimized_date`
- **Source:** dim_content
- **Bucket:** excluded
- **Role / use:** verified 91.26% null, identical rate; adjacent to a FlyRank workflow decision
- **QA status:** verified
- **Decision / reason:** excluded — high-null/unstable and workflow/optimization output risk (possible circularity)

### `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`
- **Source:** fact
- **Bucket:** excluded individually
- **Role / use:** likely too sparse per-channel
- **QA status:** not yet null-checked per column
- **Decision / reason:** excluded individually; folded into one derived ai_sessions_total instead

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.